# Day 12 - Log Transform + KFold Cross Validation


## 학습 목표
- 로그 변환으로 왜도(skewness) 감소
- KFold 교차검증 vs 단일 train/test split 비교


## 1. 데이터 로드 및 분포 확인


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

housing = fetch_california_housing()
X, y = housing.data, housing.target

print("X shape:", X.shape)
print("y skewness (original):", pd.Series(y).skew().round(4))
print("y min/max:", y.min().round(3), y.max().round(3))

# 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(y, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Original y distribution')
axes[1].hist(np.log1p(y), bins=50, color='coral', edgecolor='white')
axes[1].set_title('log1p(y) distribution')
plt.tight_layout()
plt.show()
print("y skewness (log1p):", pd.Series(np.log1p(y)).skew().round(4))


## 2. 로그 변환 전후 회귀 성능 비교


In [ ]:
def eval_ridge(X, y, name):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    model = Ridge(alpha=1.0)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    # 로그 변환한 경우 역변환
    if 'log' in name.lower():
        pred = np.expm1(pred)
        y_te_orig = np.expm1(y_te)
        r2 = r2_score(y_te_orig, pred)
        rmse = np.sqrt(mean_squared_error(y_te_orig, pred))
    else:
        r2 = r2_score(y_te, pred)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
    print(f"[{name}] R2={r2:.4f}  RMSE={rmse:.4f}")

eval_ridge(X, y, "Original y")
eval_ridge(X, np.log1p(y), "log1p(y)")


## 3. KFold 교차검증 vs 단일 split


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 단일 train/test
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model = Ridge(alpha=1.0)
model.fit(X_tr, y_tr)
single_r2 = r2_score(y_te, model.predict(X_te))
print(f"Single split R2: {single_r2:.4f}")

# KFold 5-fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(Ridge(alpha=1.0), X_scaled, y, cv=kfold, scoring='r2')
print(f"KFold 5-fold R2: mean={scores.mean():.4f}  std={scores.std():.4f}")
print(f"Individual folds: {np.round(scores, 4)}")
